# AnyCam — 04 · Step 6: does the hyperspectral model work on other cameras? (E1 + E2)

1. Re-train our Step 5 baseline (model A: ResNet18, 2x images, strong augmentation, all 204 bands) — ~25 min
2. Train three **camera-specific** models (drone MicaSense, Sentinel-2 satellite, phone RGB) — the "one model per camera" approach used today — ~25 min each
3. Test the 204-band model on **7 simulated cameras**, each: ideal / band drift / calibration+noise / both — ~15 min

Total ~2 h. Same setup: **Add Input → HyperLeaf2024 (trophy)**, **GPU T4 x2**, **Internet On**.
Best: **Save Version → Save & Run All**, then save to GitHub as `kaggle_runs/04_step6_camera_test.ipynb`.

In [1]:
import os, subprocess, glob, json, pathlib
REPO = 'https://github.com/ShahnawazQ007/one-model-many-camera.git'
if not os.path.exists('/kaggle/working/anycam'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, '/kaggle/working/anycam'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/anycam', 'pull', '-q'], check=True)
os.chdir('/kaggle/working/anycam')
!pip -q install timm tifffile imagecodecs
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - turn on the GPU accelerator!')
hits = [os.path.dirname(f) for f in glob.glob('/kaggle/input/**/train.csv', recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(f), 'images'))]
assert hits, 'Add Input -> search hyperleaf -> the one with the trophy icon'
DATA = hits[0]; print('data:', DATA, '| images:', len(glob.glob(f'{DATA}/images/*.tiff')))

GPU: Tesla T4
data: /kaggle/input/competitions/HyperLeaf2024 | images: 2410


In [2]:
# 1) the 204-band baseline (model A from Step 5)
!python experiments/train_baseline.py --data {DATA} --split official --upscale 2 --aug strong --epochs 50 --patience 12 --out runs/A_hs204

device cuda | train 1410 val 180 test 820 leaves

model.safetensors: downloading bytes:                       |  0.00B            
model.safetensors: downloading bytes: ███                   | 6.56MB            
model.safetensors: downloading bytes: ████████▊             | 18.7MB,  930kB/s  
model.safetensors: downloading bytes: ██████████████▍       | 30.7MB, 2.26MB/s  
model.safetensors: downloading bytes: ██████████████████████| 44.0MB, 4.21MB/s  
model.safetensors: reconstructing file: 100%|██████| 46.8MB / 46.8MB, 4.55MB/s  
model resnet18: 11.81M parameters
epoch  0 train 2.223 val 2.295 (79s)
epoch  1 train 1.774 val 1.710 (147s)
epoch  2 train 1.455 val 1.394 (216s)
epoch  3 train 1.211 val 1.446 (285s)
epoch  4 train 1.025 val 1.246 (354s)
epoch  5 train 0.864 val 1.304 (423s)
epoch  6 train 0.796 val 1.180 (491s)
epoch  7 train 0.707 val 1.263 (560s)
epoch  8 train 0.629 val 1.448 (630s)
epoch  9 train 0.608 val 1.166 (700s)
epoch 10 train 0.558 val 1.305 (770s)
epoch 11 trai

In [ ]:
# 2) camera-specific models: trained and tested on the same simulated camera
for cam in ['micasense_rededge_mx', 'sentinel2a_vnir', 'generic_rgb']:
    !python experiments/train_baseline.py --data {DATA} --split official --camera {cam} --upscale 2 --aug strong --epochs 50 --patience 12 --out runs/oracle_{cam}

device cuda | train 1410 val 180 test 820 leaves
model resnet18: 11.19M parameters
epoch  0 train 2.196 val 2.005 (47s)
epoch  1 train 1.770 val 1.923 (86s)
epoch  2 train 1.510 val 1.628 (124s)
epoch  3 train 1.276 val 1.566 (162s)
epoch  4 train 1.070 val 1.394 (200s)
epoch  5 train 0.942 val 1.295 (238s)
epoch  6 train 0.814 val 1.151 (275s)
epoch  7 train 0.747 val 1.324 (313s)
epoch  8 train 0.677 val 1.324 (348s)
epoch  9 train 0.656 val 1.021 (384s)


In [ ]:
# 3) test the 204-band model on all 7 cameras (+ calibration errors) and draw the figure
!python experiments/eval_cameras.py --data {DATA} --run runs/A_hs204 --oracles "runs/oracle_*" --out runs/step6_eval
from IPython.display import Image, display
display(Image('runs/step6_eval/camera_eval.png'))

## Done
When every cell has finished: **GitHub → file** `kaggle_runs/04_step6_camera_test.ipynb` → **Save**. Tell Claude "done".